# Stage 1: HDFS Data Lake Setup
## MediStream Telehealth Analytics Platform
### ISM 6562 — Final Project

This notebook documents the Stage 1 data lake foundation:
- HDFS zone verification
- Loading all 5 raw data sources into the landing zone
- Confirming file placement and replication factors

## 1. Verify HDFS is Running

In [9]:
import requests
import json

WEBHDFS = "http://namenode:9870/webhdfs/v1"

def hdfs_ls(path):
    r = requests.get(f"{WEBHDFS}{path}?op=LISTSTATUS")
    items = r.json()["FileStatuses"]["FileStatus"]
    for item in items:
        ftype = "d" if item["type"] == "DIRECTORY" else "f"
        print(f"[{ftype}] {path}/{item['pathSuffix']}  ({item['length']} bytes, replication={item['replication']})")

def hdfs_exists(path):
    r = requests.get(f"{WEBHDFS}{path}?op=GETFILESTATUS")
    return r.status_code == 200

# Test connection
r = requests.get(f"{WEBHDFS}/?op=LISTSTATUS")
print("HDFS connection status:", r.status_code)
print("Root directories:")
hdfs_ls("/")

HDFS connection status: 200
Root directories:
[d] //medistream  (0 bytes, replication=0)


## 2. Verify Zone Structure

In [10]:
print("=== LANDING ZONE ===")
hdfs_ls("/medistream/landing")

print("\n=== CURATED ZONE ===")
hdfs_ls("/medistream/curated")

print("\n=== ANALYTICS ZONE ===")
hdfs_ls("/medistream/analytics")

=== LANDING ZONE ===
[d] /medistream/landing/appointments  (0 bytes, replication=0)
[d] /medistream/landing/patient_feedback  (0 bytes, replication=0)
[d] /medistream/landing/patient_vitals  (0 bytes, replication=0)
[d] /medistream/landing/physician_schedule  (0 bytes, replication=0)
[d] /medistream/landing/session_quality  (0 bytes, replication=0)

=== CURATED ZONE ===
[d] /medistream/curated/appointments  (0 bytes, replication=0)
[d] /medistream/curated/patient_feedback  (0 bytes, replication=0)
[d] /medistream/curated/patient_vitals  (0 bytes, replication=0)
[d] /medistream/curated/physician_schedule  (0 bytes, replication=0)
[d] /medistream/curated/session_quality  (0 bytes, replication=0)

=== ANALYTICS ZONE ===
[d] /medistream/analytics/no_show_features  (0 bytes, replication=0)
[d] /medistream/analytics/patient_journey  (0 bytes, replication=0)
[d] /medistream/analytics/physician_perf  (0 bytes, replication=0)
[d] /medistream/analytics/session_quality_agg  (0 bytes, replication=

## 4. Verify All Files in Landing Zone

In [11]:
zones = [
    "/medistream/landing/appointments",
    "/medistream/landing/patient_vitals",
    "/medistream/landing/patient_feedback",
    "/medistream/landing/physician_schedule",
    "/medistream/landing/session_quality",
]

print("=== FILES IN LANDING ZONE ===")
for zone in zones:
    hdfs_ls(zone)

=== FILES IN LANDING ZONE ===
[f] /medistream/landing/appointments/appointments.csv.gz  (12084294 bytes, replication=2)
[f] /medistream/landing/patient_vitals/patient-vitals.json.gz  (6407475 bytes, replication=2)
[f] /medistream/landing/patient_feedback/patient-feedback.json.gz  (769921 bytes, replication=2)
[f] /medistream/landing/physician_schedule/physician-schedule.csv.gz  (269476 bytes, replication=2)
[f] /medistream/landing/session_quality/session-quality.csv.gz  (3829761 bytes, replication=2)


## 5. Verify Replication Factors
Expected:
- Landing zone:   RF = 3  (HIPAA-relevant raw data)
- Curated zone:   RF = 2  (cleaned, backed by landing)
- Analytics zone: RF = 1  (derived, regenerable outputs)

In [12]:
files = [
    "/medistream/landing/appointments/appointments.csv.gz",
    "/medistream/landing/patient_vitals/patient-vitals.json.gz",
    "/medistream/landing/session_quality/session-quality.csv.gz",
    "/medistream/landing/patient_feedback/patient-feedback.json.gz",
    "/medistream/landing/physician_schedule/physician-schedule.csv.gz",
]

print("=== REPLICATION FACTORS ===")
print(f"{'File':<45} {'Size':>10} {'Replication':>12}")
print("-" * 70)
for path in files:
    r = requests.get(f"{WEBHDFS}{path}?op=GETFILESTATUS")
    info = r.json()["FileStatus"]
    name = path.split("/")[-1]
    size_mb = info['length'] / 1024 / 1024
    print(f"{name:<45} {size_mb:>8.1f}MB {info['replication']:>12}")

=== REPLICATION FACTORS ===
File                                                Size  Replication
----------------------------------------------------------------------
appointments.csv.gz                               11.5MB            2
patient-vitals.json.gz                             6.1MB            2
session-quality.csv.gz                             3.7MB            2
patient-feedback.json.gz                           0.7MB            2
physician-schedule.csv.gz                          0.3MB            2


## 6. Full Zone Structure Summary

In [16]:
print("STAGE 1 SUMMARY \n")
summary = [
    ("Appointments",      "CSV.gz",  "/medistream/landing/appointments/appointments.csv.gz"),
    ("Patient Vitals",    "JSON.gz", "/medistream/landing/patient_vitals/patient-vitals.json.gz"),
    ("Session Quality",   "CSV.gz",  "/medistream/landing/session_quality/session-quality.csv.gz"),
    ("Patient Feedback",  "JSON.gz", "/medistream/landing/patient_feedback/patient-feedback.json.gz"),
    ("Physician Schedule","CSV.gz",  "/medistream/landing/physician_schedule/physician-schedule.csv.gz"),
]

print(f"{'Source':<22} {'Format':<8} {'Size':>8} {'RF':>4} ")
print("-" * 58)
for name, fmt, path in summary:
    r = requests.get(f"{WEBHDFS}{path}?op=GETFILESTATUS")
    if r.status_code == 200:
        info = r.json()["FileStatus"]
        size_mb = info['length'] / 1024 / 1024
        rf = info['replication']
        print(f"{name:<22} {fmt:<8} {size_mb:>6.1f}MB {rf:>4}")
    else:
        print(f"{name:<22} {fmt:<8} {'N/A':>8} {'N/A':>4}")

STAGE 1 SUMMARY 

Source                 Format       Size   RF 
----------------------------------------------------------
Appointments           CSV.gz     11.5MB    2
Patient Vitals         JSON.gz     6.1MB    2
Session Quality        CSV.gz      3.7MB    2
Patient Feedback       JSON.gz     0.7MB    2
Physician Schedule     CSV.gz      0.3MB    2


## 7. Data Source Summary

| Source | Format | HDFS Landing Path |
|--------|--------|-------------------|
| Appointments | CSV.gz | /medistream/landing/appointments/ |
| Patient Vitals | JSON.gz | /medistream/landing/patient_vitals/ |
| Session Quality | CSV.gz | /medistream/landing/session_quality/ |
| Patient Feedback | JSON.gz | /medistream/landing/patient_feedback/ |
| Physician Schedule | CSV.gz | /medistream/landing/physician_schedule/ |

Stage 1 complete. All raw files are in the landing zone.
Proceed to `02-spark-transforms.ipynb` for Stage 2 batch processing.